In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)



In [3]:
accounts = pd.read_csv("ravenstack_accounts.csv")
subscriptions = pd.read_csv("ravenstack_subscriptions.csv")
feature_usage = pd.read_csv("ravenstack_feature_usage.csv")
support_ticket = pd.read_csv("ravenstack_support_tickets.csv")
churn_events = pd.read_csv("ravenstack_churn_events.csv")

In [4]:
datasets = {
    "Accounts": accounts,
    "Subscriptions": subscriptions,
    "Feature Usage": feature_usage,
    "Support Tickets": support_ticket,
    "Churn Events": churn_events
}


In [5]:
accounts.drop_duplicates(inplace=True)
subscriptions.drop_duplicates(inplace=True)
feature_usage.drop_duplicates(inplace=True)
support_ticket.drop_duplicates(inplace=True)
churn_events.drop_duplicates(inplace=True)

In [7]:
for name, df in datasets.items():
  print(f"\n{name}")
  print(df.dtypes)


Accounts
account_id         object
account_name       object
industry           object
country            object
signup_date        object
referral_source    object
plan_tier          object
seats               int64
is_trial             bool
churn_flag           bool
dtype: object

Subscriptions
subscription_id      object
account_id           object
start_date           object
end_date             object
plan_tier            object
seats                 int64
mrr_amount            int64
arr_amount            int64
is_trial               bool
upgrade_flag           bool
downgrade_flag         bool
churn_flag             bool
billing_frequency    object
auto_renew_flag        bool
dtype: object

Feature Usage
usage_id               object
subscription_id        object
usage_date             object
feature_name           object
usage_count             int64
usage_duration_secs     int64
error_count             int64
is_beta_feature          bool
dtype: object

Support Tickets
ticket_id

In [12]:
accounts["signup_date"] = pd.to_datetime(accounts["signup_date"])
subscriptions["start_date"] = pd.to_datetime(subscriptions["start_date"])
subscriptions["end_date"] = pd.to_datetime(subscriptions["end_date"])
support_ticket["submitted_at"] = pd.to_datetime(support_ticket["submitted_at"])
support_ticket["closed_at"] = pd.to_datetime(support_ticket["closed_at"])
churn_events["churn_date"] = pd.to_datetime(churn_events["churn_date"])

In [14]:
text_columns = [
    "industry",
    "country",
    "referral_source"
]
for col in text_columns:
    accounts[col] = accounts[col].str.strip().str.title()

In [15]:
accounts["customer_age_days"] = (
    pd.Timestamp.today() - accounts["signup_date"]
).dt.days


In [16]:
accounts["singup_month"] = accounts["signup_date"].dt.month_name()

In [18]:
accounts["singup_quarter"] = accounts["signup_date"].dt.quarter

In [19]:
subscriptions["subscription_days"] = (
    subscriptions["end_date"] - subscriptions["start_date"]
).dt.days


In [20]:
subscriptions["arr_amount"] = subscriptions["mrr_amount"] * 12

In [21]:
subscriptions["revenue_band"] = pd.cut(
    subscriptions["mrr_amount"],
    bins = [0,100,500,1000,5000],
    labels = ["Low","Medium","High","Enterprise"]
)

In [23]:
feature_usage["heavy_user"] = (
    feature_usage["usage_count"] >= feature_usage["usage_count"].median()
)

In [24]:
support_ticket["high_priority"] = (
    support_ticket["priority"] == "High"
)


In [29]:
churn_events["refund_category"] = np.where(
    churn_events["refund_amount_usd"] > 500,
    "High Refund",
    "Low Refund"
)

In [30]:
accounts.head()

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag,customer_age_days,singup_month,singup_quarter
0,A-2e4581,Company_0,Edtech,Us,2024-10-16,Partner,Basic,9,False,False,645,October,4
1,A-43a9e3,Company_1,Fintech,In,2023-08-17,Other,Basic,18,False,True,1071,August,3
2,A-0a282f,Company_2,Devtools,Us,2024-08-27,Organic,Basic,1,False,False,695,August,3
3,A-1f0ac7,Company_3,Healthtech,Uk,2023-08-27,Other,Basic,24,True,False,1061,August,3
4,A-ce550d,Company_4,Healthtech,Us,2024-10-27,Event,Enterprise,35,False,True,634,October,4


In [31]:
subscriptions.head()

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag,subscription_days,revenue_band
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786,33432,False,False,False,True,monthly,True,111.0,Enterprise
1,S-0f6f44,A-9b9fe9,2024-06-11,NaT,Pro,17,833,9996,False,False,False,False,monthly,True,NaN,High
2,S-51c0d1,A-659280,2024-11-25,NaT,Enterprise,62,0,0,True,True,False,False,annual,False,NaN,NaN
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995,11940,False,False,False,True,monthly,True,20.0,High
4,S-cff5a2,A-ba6516,2024-01-10,NaT,Enterprise,27,5373,64476,False,False,False,False,monthly,True,NaN,NaN


In [32]:
feature_usage.head()

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature,heavy_user
0,U-1c6c24,S-0fcf7d,2023-07-27,feature_20,9,5004,0,False,False
1,U-f07cb8,S-c25263,2023-08-07,feature_5,9,369,0,False,False
2,U-096807,S-f29e7f,2023-12-07,feature_3,9,1458,0,False,False
3,U-6b1580,S-be655e,2024-07-28,feature_40,5,2085,0,False,False
4,U-720a29,S-f9b1d0,2024-12-02,feature_12,12,900,0,False,True


In [33]:
support_ticket.head()

,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag,high_priority
0,T-0024de,A-712f1c,2023-07-27,2023-07-28 03:00:00,27.0,high,74,NaN,False,False
1,T-4d04b9,A-e43bf7,2024-07-08,2024-07-09 03:00:00,27.0,urgent,144,NaN,False,False
2,T-d5e12f,A-0f3e88,2024-10-17,2024-10-17 19:00:00,19.0,urgent,93,4.0,False,False
3,T-dfce9a,A-4c56c9,2024-09-08,2024-09-09 23:00:00,47.0,medium,126,5.0,False,False
4,T-c59f77,A-6f8ad2,2024-11-30,2024-12-01 02:00:00,26.0,medium,8,NaN,False,False


In [34]:
churn_events.head()

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text,refund_category
0,C-816288,A-c37cab,2024-10-27,pricing,4.03,False,False,False,switched to competitor,Low Refund
1,C-5a81e7,A-37f969,2024-06-25,support,96.45,True,False,False,NaN,Low Refund
2,C-a174be,A-b07346,2024-11-12,budget,0.00,False,False,False,missing features,Low Refund
3,C-accb39,A-1e50e0,2023-11-01,budget,54.94,False,False,False,switched to competitor,Low Refund
4,C-92f889,A-956988,2024-12-30,unknown,0.00,False,True,True,too expensive,Low Refund


In [35]:
accounts.to_csv("accounts_cleaned.csv", index=False)
subscriptions.to_csv("subscriptions_cleaned.csv", index=False)
feature_usage.to_csv("feature_usage_cleaned.csv", index=False)
support_ticket.to_csv("support_ticket_cleaned.csv", index=False)

In [36]:
feature_usage.to_csv("feature_usage_cleaned.csv", index=False)
support_ticket.to_csv("support_ticket_cleaned.csv", index=False)
churn_events.to_csv("churn_events_clean.csv", index = False )

In [ ]:
churn_events.to_csv("churn_events_clean.csv", index = False )